In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
paths = get_paths(ROOT)
paths


ProjectPaths(root=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526'), src=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/src'), notebooks=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/notebooks'), data_processed=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed'), data_raw=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw'), checkpoints=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/checkpoints'))

In [4]:
from pathlib import Path
import pandas as pd

train_chunks_path = paths.data_processed / "train_80k_chunks.parquet"
val_chunks_path   = paths.data_processed / "val_80k_chunks.parquet"
test_chunks_path  = paths.data_processed / "test_80k_chunks.parquet"
train_counts_path = paths.data_processed / "train_120k_chunk_counts.parquet"

assert train_chunks_path.exists()
assert val_chunks_path.exists()
assert test_chunks_path.exists()
assert train_counts_path.exists()

(train_chunks_path, val_chunks_path, test_chunks_path, train_counts_path)


(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/test_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_120k_chunk_counts.parquet'))

In [5]:
from teacher_finetune import load_chunks_dataset

train_ds = load_chunks_dataset(
    chunks_parquet=train_chunks_path,
    chunk_counts_parquet=train_counts_path,
    add_sample_weight=True,
)
val_ds = load_chunks_dataset(
    chunks_parquet=val_chunks_path,
    add_sample_weight=False,
)

train_ds, val_ds


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['review_id', 'movie_id', 'labels', 'chunk_index', 'input_ids', 'attention_mask', 'token_type_ids', 'sample_weight'],
     num_rows: 212943
 }),
 Dataset({
     features: ['review_id', 'movie_id', 'labels', 'chunk_index', 'input_ids', 'attention_mask', 'token_type_ids'],
     num_rows: 26663
 }))

In [6]:
import torch
from teacher_finetune import (
    TeacherModelConfig,
    build_teacher_tokenizer,
    build_teacher_model,
    build_collator,
    bf16_supported,
)

MODEL_NAME = "bert-large-uncased"

tokenizer = build_teacher_tokenizer(MODEL_NAME)
collator = build_collator(tokenizer)

model_cfg = TeacherModelConfig(
    model_name=MODEL_NAME,
    gradient_checkpointing=False,  # metti True solo se OOM
)

model = build_teacher_model(model_cfg)

print("CUDA:", torch.cuda.is_available(), "| bf16_supported:", bf16_supported())


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CUDA: True | bf16_supported: True


In [ ]:
from transformers import TrainingArguments
from teacher_finetune import WeightedBCETrainer

OUT_DIR = paths.checkpoints / "teacher_ft_120k"

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=8,  #4 if OOM
    per_device_eval_batch_size=4,   #8 if OOM
    gradient_accumulation_steps=8,     # eff batch 32
    learning_rate=2e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=bf16_supported(),
    fp16=not bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = WeightedBCETrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(OUT_DIR / "final"))


C:\Users\cola0\AppData\Local\Temp\ipykernel_17260\223002473.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedBCETrainer(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 